In [1]:
# ======================================================
# NOTEBOOK: PROCESS "THE COUNT OF MONTE CRISTO" (H100 VERSION)
# ======================================================

In [2]:
!pip install -U "transformers>=4.40.0" accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 24.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


In [3]:
import os
import re
import pandas as pd
import pickle
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoConfig
from tqdm import tqdm

In [4]:
# --- 1. CONFIG ---
BOOK_NAME = "The Count of Monte Cristo"
FILENAME = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/The Count of Monte Cristo.txt" 
OUTPUT_PKL = "montecristo_chunks.pkl"
OUTPUT_EMB = "montecristo_embeddings_qwen.pkl"

# SOTA Model for H100
MODEL_ID = "Alibaba-NLP/gte-Qwen2-7B-instruct" 

CHUNK_SIZE = 600
OVERLAP = 150

# Character list for Metadata
CHARACTERS = [
    "Dantes", "Edmond", "Monte Cristo", "Faria", "Mercedes", "Fernand", 
    "Villefort", "Danglars", "Caderousse", "Albert", "Haydee", "Noirtier", 
    "Morrel", "Bertuccio", "Benedetto", "Andrea", "Valentine", "Maximilian"
]

In [5]:
# --- 2. MODEL HELPER FUNCTIONS ---

def last_token_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

def get_embeddings(texts, model, tokenizer, batch_size=4):
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i:i + batch_size]
        
        batch_dict = tokenizer(
            batch_texts, 
            max_length=8192, 
            padding=True, 
            truncation=True, 
            return_tensors='pt'
        ).to("cuda")

        with torch.no_grad():
            # use_cache=False is CRITICAL to avoid "DynamicCache" errors
            outputs = model(**batch_dict, use_cache=False)
            embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
            embeddings = F.normalize(embeddings, p=2, dim=1)
            
        all_embeddings.append(embeddings.cpu().numpy())
        
    return [item for sublist in all_embeddings for item in sublist]

In [6]:
# --- 3. SPECIFIC CLEANER FOR MONTE CRISTO ---
def clean_montecristo_text(text):
    print("Cleaning text...")
    
    # 1. Start Marker
    start_match = re.search(r"Chapter 1\. Marseilles", text)
    if start_match:
        text = text[start_match.start():]
        print("-> Found Start of Chapter 1.")
    else:
        print("-> WARNING: Chapter 1 start not found.")
    
    # 2. End Marker
    end_match = re.search(r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK", text)
    if end_match:
        text = text[:end_match.start()]
        print("-> Found End Marker.")
    
    # 3. Noise Removal
    # Remove volume headers like "VOLUME ONE"
    text = re.sub(r"VOLUME [A-Z]+", "", text)
    # Remove page numbers like 0207m (specific to this text file)
    text = re.sub(r"\d+m\s+", "", text)
    
    return text

def get_metadata(chunk_text):
    found = [c for c in CHARACTERS if c in chunk_text]
    if found: return f"Characters: {', '.join(found)}"
    return ""

def process_book(filename):
    if not os.path.exists(filename):
        raise FileNotFoundError(f"File {filename} not found! Upload it first.")
        
    with open(filename, 'r', encoding='utf-8') as f:
        raw_text = f.read()
        
    clean_text = clean_montecristo_text(raw_text)
    words = clean_text.split()
    chunks = []
    
    # Regex: "Chapter 1.", "Chapter 10."
    chapter_pattern = re.compile(r"^Chapter\s+\d+\.")
    current_chapter = "Chapter 1"
    
    print("Chunking...")
    for i in range(0, len(words), CHUNK_SIZE - OVERLAP):
        chunk_words = words[i:i + CHUNK_SIZE]
        chunk_text = " ".join(chunk_words)
        
        start_snippet = " ".join(chunk_words[:10])
        match = chapter_pattern.search(start_snippet)
        if match:
            current_chapter = match.group(0)
            
        meta = get_metadata(chunk_text)
        # SOTA Index format: Meta + Text
        search_text = f"{meta} {chunk_text}" 
        
        chunks.append({
            "text": chunk_text,
            "book_name": BOOK_NAME,
            "chapter": current_chapter,
            "metadata": meta,
            "search_text": search_text, 
            "chunk_index": i
        })
        
    return chunks

In [7]:
# --- 4. EXECUTION ---

# A. Load Model (Robust)
print(f"Loading Model: {MODEL_ID}...")
try:
    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    config.use_cache = False  # Disable KV cache for embeddings
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        MODEL_ID, 
        config=config,
        trust_remote_code=True,
        torch_dtype=torch.float16
    ).to("cuda")
    model.eval()
    print("Model loaded successfully.")
except Exception as e:
    print(f"CRITICAL ERROR loading model: {e}")
    raise e

Loading Model: Alibaba-NLP/gte-Qwen2-7B-instruct...


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct:
- tokenization_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00004-of-00007.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00005-of-00007.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00007.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Model loaded successfully.


In [8]:
# B. Process
print(f"Processing {FILENAME}...")
data = process_book(FILENAME)
df = pd.DataFrame(data)
print(f"Generated {len(df)} chunks.")

Processing /kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/The Count of Monte Cristo.txt...
Cleaning text...
-> Found Start of Chapter 1.
-> Found End Marker.
Chunking...
Generated 1024 chunks.


In [9]:
# C. Embed
print("Generating Embeddings...")
embeddings = get_embeddings(df['search_text'].tolist(), model, tokenizer)

Generating Embeddings...


Embedding: 100%|██████████| 256/256 [00:28<00:00,  8.89it/s]


In [10]:
# D. Save
df.to_pickle(OUTPUT_PKL)
with open(OUTPUT_EMB, "wb") as f:
    pickle.dump(embeddings, f)

print(f"DONE! Saved {OUTPUT_PKL} and {OUTPUT_EMB}")

DONE! Saved montecristo_chunks.pkl and montecristo_embeddings_qwen.pkl
